# Projet — Exploitation du dateset CL-Drive

## Estimation de la charge cognitive du conducteur

Ce TP vient après les TD1, TD2, TD3 et TD4.

Les TD ont déjà permis de travailler :

- la compréhension du papier CL-Drive ;
- le protocole expérimental ;
- la segmentation en fenêtres de 10 s ;
- le prétraitement EEG ;
- l'extraction des features EEG.

Le point de départ du TP est donc le dossier généré à la fin du TD4 :

```text
EEG_Features_10s/
```

Ce TP ne revient pas sur le calcul des features. Il exploite les features déjà extraites pour construire un pipeline d'apprentissage automatique.

## Objectif du TP

Construire un pipeline complet :

```text
EEG_Features_10s
→ Normalized_Features_10s/EEG
→ Normalized_Features_10s_With_Label/EEG
→ Dataset EEG supervisé
→ Classification de la charge cognitive
→ Évaluation
→ Interprétation
```

Dans un premier temps, on se limite à l'EEG uniquement.

La multimodalité, c'est-à-dire l'ajout de ECG, EDA et Gaze, sera proposée uniquement comme extension à la fin du sujet.

## 1. Structure attendue des dossiers

Avant de commencer, le dossier de travail doit contenir au minimum :

```text
CL-Drive/
├── EEG/<participant_id>/
│   ├── eeg_data_level_1.csv ... eeg_data_level_9.csv
│   ├── eeg_data_baseline.csv
│   └── filtered_eeg_data_level_*.csv  (générés par TD3)
│
├── EEG_Features/
│   ├── features_<pid>.csv          (générés par TD4, données de tâche)
│   └── baseline_features_<pid>.csv (générés par TD4, données baseline)
│
├── Labels/
│   ├── 1030.csv
│   └── ...
```

Le TP va générer deux nouveaux dossiers :

```text
CL-Drive/
├── Normalized_Features/EEG/
│   └── norm_features_<pid>.csv
│
└── Normalized_Features_With_Label/EEG/
    └── norm_features_<pid>.csv   (avec colonnes Level et Label)
```

## Question

Pourquoi ne faut-il pas entraîner directement les modèles sur les fichiers `EEG_Features` ?

### Réponse

Il ne faut pas entraîner directement sur les features brutes pour deux raisons principales :

1. **Variabilité inter-sujets** : Les signaux EEG varient fortement d'un individu à l'autre (anatomie, impédance des électrodes, activité de repos). Sans normalisation par la baseline, les features reflètent autant les différences entre sujets que la charge cognitive réelle. Le modèle risquerait d'apprendre à « reconnaître » les sujets plutôt qu'à détecter la charge.

2. **Fuite de données (data leakage)** : La standardisation z-score doit être calculée *après* la séparation train/test. Si on normalise tout le dataset d'un coup, les statistiques du test « contaminent » le train, ce qui produit des estimations de performance artificiellement trop optimistes.

In [29]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Chemin de base — adapter si nécessaire
BASE_PATH = Path(r"c:\Users\pault\OneDrive\Documents\neurone\Neuronanale\CL-Drive")

EEG_FEATURE_DIR    = BASE_PATH / "EEG_Features"
LABEL_DIR          = BASE_PATH / "Labels"
NORMALIZED_ROOT    = BASE_PATH / "Normalized_Features"
NORMALIZED_EEG_DIR = NORMALIZED_ROOT / "EEG"
LABELED_ROOT       = BASE_PATH / "Normalized_Features_With_Label"
LABELED_EEG_DIR    = LABELED_ROOT / "EEG"

NORMALIZED_EEG_DIR.mkdir(parents=True, exist_ok=True)
LABELED_EEG_DIR.mkdir(parents=True, exist_ok=True)

# Colonnes de métadonnées 
METADATA_COLUMNS = ["participant_id", "scenario", "window_idx", "canal",
                    "t_start_s", "t_end_s"]

# Vérification de l'état des données
task_files     = list(EEG_FEATURE_DIR.glob("features_*.csv"))
baseline_files = list(EEG_FEATURE_DIR.glob("baseline_features_*.csv"))
print(f"Fichiers de features de tâche    : {len(task_files)}")
print(f"Fichiers de features baseline    : {len(baseline_files)}")
if not baseline_files:
    print("\n[INFO] Aucun fichier baseline trouvé.")
    print("       La normalisation par baseline sera ignorée.")
    print("       Pour l'activer : exécutez le batch complet dans TD3 et TD4.")

Fichiers de features de tâche    : 1
Fichiers de features baseline    : 0

[INFO] Aucun fichier baseline trouvé.
       La normalisation par baseline sera ignorée.
       Pour l'activer : exécutez le batch complet dans TD3 et TD4.


## 2. Normalisation des features EEG

À la fin du TD4, chaque fichier CSV contient des features EEG calculées sur des fenêtres de 10 secondes.

La normalisation doit suivre deux étapes :

### Étape 1 — Normalisation par la baseline du sujet

Pour chaque sujet, les fichiers de baseline servent à calculer une valeur moyenne de référence pour chaque feature :

$$
\mu_{baseline}^{(s,f)} = \frac{1}{N}\sum_{i=1}^{N} x_i^{(s,f)}
$$

où :

- $s$ désigne le sujet ;
- $f$ désigne la feature ;
- $x_i^{(s,f)}$ désigne la valeur de la feature pendant la baseline.

Chaque valeur de feature dans les fichiers de tâche est ensuite divisée par la moyenne de baseline correspondante :

$$
x_{norm}^{(s,f)} = \frac{ x^{(s,f)} }{ \mu_{baseline}^{(s,f)} }
$$

### Étape 2 — Standardisation z-score

On applique ensuite une standardisation :

$$
z = \frac{x - \mu}{\sigma}
$$

Cela permet d'obtenir des features centrées et réduites. Cette étape devra toutefois être réalisée plus loin dans le pipeline, après la séparation des données entre les ensembles d'entraînement et de test (voir section 8 ci-dessous).

## Question

Quel est l'intérêt de la normalisation par baseline dans des signaux physiologiques ?

### Réponse

La normalisation par baseline présente deux avantages majeurs pour les signaux physiologiques :

1. **Correction de la variabilité inter-sujets** : Chaque conducteur possède un niveau d'activité EEG de repos différent. En divisant par la baseline, on exprime chaque valeur comme un *ratio relatif à l'état de repos du sujet*. Cela rend les features comparables d'un individu à l'autre, même si leurs niveaux absolus diffèrent.

2. **Isolement du signal de tâche** : La baseline capture les caractéristiques « statiques » du sujet (activité cérébrale de fond, bruit de mesure persistant, artefacts liés au montage). La division par baseline supprime cette composante statique et ne conserve que les *variations induites par la charge cognitive*.

In [16]:
def get_feature_columns(df, metadata_columns=None):
    """Retourne les colonnes correspondant aux features (tout sauf les métadonnées)."""
    if metadata_columns is None:
        metadata_columns = METADATA_COLUMNS
    return [c for c in df.columns if c not in metadata_columns]


def compute_baseline_averages(feature_dir=None):
    """
    Calcule, pour chaque participant et canal, la moyenne de chaque feature sur la baseline.

    Retourne un dict imbriqué : {participant_id: {canal: {feature: valeur_moyenne}}}
    """
    if feature_dir is None:
        feature_dir = EEG_FEATURE_DIR
    baseline_avgs = {}
    for fpath in sorted(Path(feature_dir).glob("baseline_features_*.csv")):
        df = pd.read_csv(fpath)
        pid = df["participant_id"].iloc[0]
        feat_cols = get_feature_columns(df)
        # Moyenne par canal sur toutes les fenêtres de baseline
        baseline_avgs[pid] = (
            df.groupby("canal")[feat_cols].mean().to_dict(orient="index")
        )
    return baseline_avgs


def normalize_by_baseline(df, participant_id, baseline_avgs):
    """
    Divise chaque feature par sa moyenne de baseline (canal par canal).
    Si la baseline est absente pour ce participant, retourne le DataFrame inchangé.
    """
    if participant_id not in baseline_avgs:
        return df.copy()
    df_norm = df.copy()
    feat_cols = get_feature_columns(df_norm)
    for canal, canal_means in baseline_avgs[participant_id].items():
        mask = df_norm["canal"] == canal
        for feat in feat_cols:
            mean_val = canal_means.get(feat, np.nan)
            if pd.notna(mean_val) and mean_val != 0:
                df_norm.loc[mask, feat] = df_norm.loc[mask, feat] / mean_val
            else:
                df_norm.loc[mask, feat] = np.nan
    return df_norm


def run_eeg_normalization():
    """
    Génère les fichiers du dossier Normalized_Features/EEG
    à partir du dossier EEG_Features.
    """
    baseline_avgs = compute_baseline_averages()
    if baseline_avgs:
        print(f"Baseline trouvée pour {len(baseline_avgs)} sujet(s).")
    else:
        print("[INFO] Aucune baseline — features copiées sans normalisation par baseline.")

    task_files = sorted(EEG_FEATURE_DIR.glob("features_*.csv"))
    print(f"Traitement de {len(task_files)} fichier(s) de features...")

    for fpath in task_files:
        df = pd.read_csv(fpath)
        pid = df["participant_id"].iloc[0]
        df_norm = normalize_by_baseline(df, pid, baseline_avgs)
        out_path = NORMALIZED_EEG_DIR / f"norm_features_{pid}.csv"
        df_norm.to_csv(out_path, index=False)
        print(f"  Sauvegardé : {out_path.name}")

    print("Normalisation terminée.")

In [17]:
run_eeg_normalization()

[INFO] Aucune baseline — features copiées sans normalisation par baseline.
Traitement de 1 fichier(s) de features...
  Sauvegardé : norm_features_1030.csv
Normalisation terminée.


## 3. Vérification du dossier `Normalized_Features/EEG`

Après exécution de la normalisation, vérifiez que le dossier contient bien des fichiers `norm_*.csv`.

## Question

Pourquoi les fichiers de baseline ne sont-ils pas copiés dans le dossier normalisé final ?

### Réponse

Les fichiers de baseline ne sont pas copiés pour trois raisons :

1. **Rôle de référence uniquement** : Les fichiers baseline servent uniquement à calculer les moyennes μ_baseline utilisées pour la normalisation. Ils ne représentent pas des situations de conduite avec charge cognitive.

2. **Pas de label PAAS** : Le conducteur est au repos pendant la baseline — aucun score PAAS n'est associé à ces fenêtres. Sans cible d'apprentissage, ces lignes ne peuvent pas servir à entraîner ou évaluer un classifieur.

3. **Pollution du dataset** : Les inclure créerait des lignes sans label dans le dataset supervisé, ce qui perturberait l'entraînement des modèles.

In [18]:
# Vérification du dossier Normalized_Features/EEG
norm_files = sorted(NORMALIZED_EEG_DIR.glob("norm_features_*.csv"))
print(f"Fichiers normalisés : {len(norm_files)}")
for fpath in norm_files:
    df_check = pd.read_csv(fpath)
    feat_cols = get_feature_columns(df_check)
    print(f"  {fpath.name} : {len(df_check)} lignes, {len(feat_cols)} features")
    print(f"    Canaux : {sorted(df_check['canal'].unique().tolist())}")
    print(f"    Scénarios : {sorted(df_check['scenario'].unique().tolist())}")

Fichiers normalisés : 1
  norm_features_1030.csv : 568 lignes, 40 features
    Canaux : ['AF7', 'AF8', 'TP10', 'TP9']
    Scénarios : [1, 2, 3, 5, 6, 7, 8, 9]


## 4. Ajout des colonnes `Level` et `Label`

Les fichiers normalisés ne contiennent pas encore la cible d'apprentissage.

Il faut maintenant associer chaque fenêtre de 10 secondes à son score PAAS.

Les labels sont stockés dans le dossier :

```text
Labels/
```

Chaque fichier de labels correspond à un sujet, par exemple :

```text
Labels/ID1.csv
Labels/ID2.csv
...
```

Dans ces fichiers, on suppose une structure du type :

| time | lvl_1 | lvl_2 | ... | lvl_9 |
|---:|---:|---:|---|---:|
| 10 | 2 | 3 | ... | 5 |
| 20 | 2 | 4 | ... | 6 |
| ... | ... | ... | ... | ... |

Pour une fenêtre d'indice `Window`, le temps associé est :

$$
time = (Window + 1) \times 10
$$

Le niveau du scénario est extrait du nom du fichier avec une expression régulière :

```text
level_1 → Level = 1
level_2 → Level = 2
...
level_9 → Level = 9
```

Le score PAAS est ensuite récupéré dans la colonne :

```text
lvl_<Level>
```

Exemple : si `Level = 4`, on lit la colonne `lvl_4`.


In [ ]:
def extract_level_from_filename(file_name):
    """
    Extrait le numéro de scénario depuis un nom de fichier.
    Exemple : norm_filtered_level_3.csv → 3
    Note : dans notre structure, le niveau est directement dans la colonne 'scenario'.
    """
    m = re.search(r'level[_-](\d+)', str(file_name), re.IGNORECASE)
    return int(m.group(1)) if m else None


def get_label_for_row(row, labels_df):
    """
    Retourne le score PAAS correspondant à une ligne de features.

    - window_idx → time_stamp = (window_idx + 1) * 10
    - scenario   → colonne lvl_{scenario} dans labels_df
    """
    time_stamp = (int(row["window_idx"]) + 1) * 10
    level = int(row["scenario"])
    label_col = f"lvl_{level}"
    if label_col not in labels_df.columns:
        return np.nan
    match = labels_df.loc[labels_df["time"] == time_stamp, label_col]
    return float(match.iloc[0]) if len(match) > 0 else np.nan


def attach_labels_eeg():
    """
    Génère les fichiers du dossier Normalized_Features_With_Label/EEG.
    Ajoute les colonnes Level (numéro de scénario) et Label (score PAAS).
    """
    norm_files = sorted(NORMALIZED_EEG_DIR.glob("norm_features_*.csv"))
    print(f"Attachement des labels pour {len(norm_files)} fichier(s)...")

    for fpath in norm_files:
        df = pd.read_csv(fpath)
        pid = df["participant_id"].iloc[0]

        label_path = LABEL_DIR / f"{pid}.csv"
        if not label_path.exists():
            print(f"  [AVERTISSEMENT] Label introuvable pour {pid} — skip.")
            continue
        labels_df = pd.read_csv(label_path)

        df["Level"] = df["scenario"].astype(int)    
        df["Label"] = df.apply(lambda row: get_label_for_row(row, labels_df), axis=1)

        n_before = len(df)
        df = df.dropna(subset=["Label"])
        df["Label"] = df["Label"].astype(int)
        n_dropped = n_before - len(df)
        if n_dropped:
            print(f"  {fpath.name} : {n_dropped} ligne(s) sans label supprimée(s).")

        out_path = LABELED_EEG_DIR / f"norm_features_{pid}.csv"
        df.to_csv(out_path, index=False)
        print(f"  Sauvegardé : {out_path.name}")

    print("Terminé.")

In [20]:
attach_labels_eeg()

Attachement des labels pour 1 fichier(s)...
  Sauvegardé : norm_features_1030.csv
Terminé.


## 5. Vérification du dossier `Normalized_Features_With_Label/EEG`

Le dossier final doit contenir des fichiers CSV avec au moins :

- les métadonnées : `participant_id`, `scenario`, `window_idx`, `canal`, `t_start_s`, `t_end_s` ;
- les features EEG normalisées ;
- la colonne `Level` ;
- la colonne `Label`.

## Question

Quelle est la différence entre `Level` et `Label` dans ce TP ? Pourquoi faut-il ajouter à la fois `Level` et `Label` ?

### Réponse

- **`Level`** désigne le **numéro de scénario** (1 à 9). Il identifie dans quelle condition de conduite la fenêtre a été enregistrée (ex. : autoroute, route nationale, etc.). C'est un identifiant de condition expérimentale.

- **`Label`** désigne le **score PAAS** (1 à 9) recueilli pour ce sujet à ce moment précis du scénario. C'est la mesure subjective de la charge cognitive ressentie par le conducteur. C'est la **cible d'apprentissage** pour les classifieurs.

Les deux colonnes sont nécessaires :
- `Level` est utilisé comme **clé de jointure** pour trouver la bonne colonne dans le fichier de labels (`lvl_1`, `lvl_2`, ...). Sans lui, on ne saurait pas quelle colonne lire.
- `Label` est la **valeur de prédiction** que le modèle doit apprendre à estimer. C'est lui qui sera binarisé en classe faible/élevée.

In [21]:
# Vérification du dossier Normalized_Features_With_Label/EEG
labeled_files = sorted(LABELED_EEG_DIR.glob("norm_features_*.csv"))
print(f"Fichiers labellisés : {len(labeled_files)}")
for fpath in labeled_files:
    df_check = pd.read_csv(fpath)
    labels_present = sorted(df_check["Label"].unique().tolist())
    scenarios = sorted(df_check["scenario"].unique().tolist())
    print(f"  {fpath.name} : {len(df_check)} lignes")
    print(f"    Scores PAAS présents : {labels_present}")
    print(f"    Scénarios : {scenarios}")
    print(f"    Colonnes : {df_check.columns.tolist()[-5:]}")

Fichiers labellisés : 1
  norm_features_1030.csv : 568 lignes
    Scores PAAS présents : [1, 2, 3, 4, 5, 6, 7, 8, 9]
    Scénarios : [1, 2, 3, 5, 6, 7, 8, 9]
    Colonnes : ['raw_median', 'raw_var', 'raw_std', 'Level', 'Label']


## 6. Construction du dataset EEG supervisé

Une fois les fichiers normalisés et labellisés générés, on peut les concaténer pour construire un tableau unique.

Chaque ligne représente une fenêtre EEG de 10 secondes pour un canal.

On construit ensuite deux problèmes possibles :

### Classification binaire

| Score PAAS | Classe |
|---:|---|
| 1 à 4 | faible |
| 5 à 9 | élevée |

### Classification ternaire, extension

| Score PAAS | Classe |
|---:|---|
| 1 à 3 | faible |
| 4 à 6 | moyenne |
| 7 à 9 | élevée |

Dans ce TP, l'objectif principal est la classification binaire.

In [22]:
def load_labeled_eeg_dataset():
    """Concatène tous les fichiers CSV de Normalized_Features_With_Label/EEG."""
    files = sorted(LABELED_EEG_DIR.glob("norm_features_*.csv"))
    if not files:
        raise FileNotFoundError(f"Aucun fichier trouvé dans {LABELED_EEG_DIR}")
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)


df = load_labeled_eeg_dataset()
print(df.shape)
df.head()

(568, 48)


,participant_id,scenario,window_idx,canal,t_start_s,t_end_s,delta_abs,delta_mean,delta_max,delta_min,...,lempel_ziv,higuchi_fd,raw_mean,raw_min,raw_max,raw_median,raw_var,raw_std,Level,Label
0,1030,1,0,TP9,0.0,9.996094,1707.172283,243.881755,374.742959,149.928056,...,0.588210,0.590767,0.280324,-221.788205,156.385686,4.863292,1522.258232,39.016128,1,2
1,1030,1,0,AF7,0.0,9.996094,281.720600,40.245800,84.740594,13.890115,...,0.645704,0.654996,-0.617331,-70.960950,71.100002,-0.289262,276.770812,16.636430,1,2
2,1030,1,0,AF8,0.0,9.996094,568.265371,81.180767,103.084941,39.904480,...,0.619168,0.653607,0.561979,-77.242674,132.932700,0.394474,539.872354,23.235153,1,2
3,1030,1,0,TP10,0.0,9.996094,1400.509939,200.072848,318.348794,94.773375,...,0.667817,0.642326,-0.344195,-232.465476,134.956839,2.230255,1307.653665,36.161494,1,2
4,1030,1,1,TP9,10.0,19.996094,2999.649103,428.521300,514.169096,310.365244,...,0.552829,0.512821,0.112039,-287.953060,146.757376,3.611857,2310.475011,48.067401,1,2


In [23]:
# Classification binaire : PAAS ≤ 4 → 0 (charge faible), PAAS ≥ 5 → 1 (charge élevée)
df["Label_Binary"] = (df["Label"] >= 5).astype(int)

# Extension ternaire : 1-3 → 0 (faible), 4-6 → 1 (moyenne), 7-9 → 2 (élevée)
df["Label_Ternary"] = df["Label"].apply(
    lambda s: 0 if s <= 3 else (1 if s <= 6 else 2)
)

print("Distribution binaire :")
counts_bin = df["Label_Binary"].value_counts().sort_index()
for k, v in counts_bin.items():
    label_str = "Faible" if k == 0 else "Elevee"
    print(f"  {k} ({label_str}) : {v} ({100*v/len(df):.1f}%)")

print("\nDistribution ternaire :")
counts_ter = df["Label_Ternary"].value_counts().sort_index()
noms = {0: "Faible", 1: "Moyenne", 2: "Elevee"}
for k, v in counts_ter.items():
    print(f"  {k} ({noms[k]}) : {v} ({100*v/len(df):.1f}%)")

Distribution binaire :
  0 (Faible) : 148 (26.1%)
  1 (Elevee) : 420 (73.9%)

Distribution ternaire :
  0 (Faible) : 128 (22.5%)
  1 (Moyenne) : 184 (32.4%)
  2 (Elevee) : 256 (45.1%)


## 7. Préparation de la matrice d'apprentissage

On doit séparer :

- les métadonnées ;
- les features numériques EEG ;
- la cible d'apprentissage.

Les 4 canaux EEG (AF7, AF8, TP9, TP10) sont **pivotés** pour obtenir une seule ligne par fenêtre.
Chaque ligne représente donc une fenêtre de 10 s avec ses 4 × 40 = **160 features**.

## Question

Pourquoi ne faut-il pas inclure `participant_id`, `scenario`, `window_idx`, `Level` ou `Label` dans les features du modèle ?

### Réponse

- **`participant_id`** : Identifiant du sujet. L'inclure permettrait au modèle de mémoriser les patterns propres à chaque individu au lieu d'apprendre la charge cognitive. En généralisation sur un nouveau conducteur (LOSO), le modèle n'aurait jamais vu cet identifiant et ne pourrait pas l'utiliser.

- **`scenario`** : Numéro de condition expérimentale. Certains scénarios sont intrinsèquement plus chargés (ex. : route urbaine vs. autoroute). Si on l'inclut, le modèle apprend « le scénario 5 est chargé » au lieu d'apprendre depuis les signaux EEG.

- **`window_idx`** : Index temporel dans le scénario. Ce n'est pas un signal physiologique — il introduirait une dépendance à l'ordre des fenêtres, non généralisable.

- **`Level`** et **`Label`** : Ce sont les **cibles** (ou directement dérivées de la cible). Les inclure dans les features crée une fuite de données directe : le modèle « tricherait » en connaissant déjà la réponse.

In [24]:
# Préparation de la matrice d'apprentissage
# Pivot : 4 canaux × 40 features = 160 features par fenêtre

EXTRA_COLS = ["Level", "Label", "Label_Binary", "Label_Ternary"]
feat_cols = [c for c in df.columns if c not in METADATA_COLUMNS + EXTRA_COLS]

canaux = sorted(df["canal"].unique())
sub_dfs = []
for canal in canaux:
    df_c = df[df["canal"] == canal].copy()
    # Renommer les features avec le suffixe du canal
    rename = {f: f"{f}__{canal}" for f in feat_cols}
    df_c = df_c.rename(columns=rename)
    feat_renamed = list(rename.values())
    keep = ["participant_id", "scenario", "window_idx",
            "Label_Binary", "Label_Ternary"] + feat_renamed
    sub_dfs.append(df_c[keep].set_index(["participant_id", "scenario", "window_idx"]))

# Joindre les 4 canaux sur (participant_id, scenario, window_idx)
df_wide = sub_dfs[0]
for other in sub_dfs[1:]:
    df_wide = df_wide.join(
        other.drop(columns=["Label_Binary", "Label_Ternary"]),
        how="inner"
    )
df_wide = df_wide.reset_index()

feat_cols_wide = [c for c in df_wide.columns
                  if c not in ["participant_id", "scenario", "window_idx",
                                "Label_Binary", "Label_Ternary"]]

X = df_wide[feat_cols_wide].values.astype(float)
y = df_wide["Label_Binary"].values.astype(int)
groups = df_wide["participant_id"].values  # utilisé pour LOSO

print("Nombre d'exemples :", X.shape[0])
print("Nombre de features :", X.shape[1])
print("Exemples de features :", feat_cols_wide[:5])
print("Distribution des classes :", dict(zip(*np.unique(y, return_counts=True))))
print("Sujets disponibles :", np.unique(groups).tolist())

Nombre d'exemples : 142
Nombre de features : 160
Exemples de features : ['delta_abs__AF7', 'delta_mean__AF7', 'delta_max__AF7', 'delta_min__AF7', 'delta_median__AF7']
Distribution des classes : {np.int64(0): np.int64(37), np.int64(1): np.int64(105)}
Sujets disponibles : [1030]


## 8. Classification EEG — premiers modèles

On teste plusieurs modèles classiques :

- LDA ;
- SVM ;
- Random Forest ;
- KNN ;
- Naive Bayes ;
- Decision Tree ;
- AdaBoost ;
- MLP.

La normalisation `StandardScaler` est placée dans le `sklearn.pipeline.Pipeline` pour éviter une fuite de données entre apprentissage et test. Il faut ajuster le `StandardScaler` uniquement sur les données d’entraînement :

`scaler.fit_transform(X_train)`

Puis appliquer la transformation aux données de test avec :

`scaler.transform(X_test)`

In [25]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import f1_score, make_scorer

classifiers = {
    "LDA":           LinearDiscriminantAnalysis(),
    "SVM":           SVC(kernel="rbf", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "KNN":           KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes":   GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "AdaBoost":      AdaBoostClassifier(n_estimators=50, random_state=42),
    "MLP":           MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42),
}

# Pipeline : StandardScaler intégré pour éviter toute fuite entre train et test
pipelines = {
    name: Pipeline([("scaler", StandardScaler()), ("clf", clf)])
    for name, clf in classifiers.items()
}

# 10-fold cross-validation stratifiée
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scoring = {
    "accuracy": "accuracy",
    "f1":       make_scorer(f1_score, zero_division=0),
}

results_cv = {}
print("=== 10-fold Cross-Validation stratifiée ===\n")
print(f"  {'Modèle':<20} {'Accuracy':>10} {'F1-score':>10}")
print("  " + "-" * 42)
for name, pipe in pipelines.items():
    cv = cross_validate(pipe, X, y, cv=skf, scoring=scoring, n_jobs=-1)
    results_cv[name] = {
        "accuracy": cv["test_accuracy"].mean(),
        "f1":       cv["test_f1"].mean(),
    }
    print(f"  {name:<20} {results_cv[name]['accuracy']:>10.3f} {results_cv[name]['f1']:>10.3f}")

best_cv = max(results_cv, key=lambda k: results_cv[k]["f1"])
print(f"\n  Meilleur modèle (F1) : {best_cv} — F1 = {results_cv[best_cv]['f1']:.3f}")

=== 10-fold Cross-Validation stratifiée ===

  Modèle                 Accuracy   F1-score
  ------------------------------------------
  LDA                       0.669      0.750
  SVM                       0.880      0.921
  Random Forest             0.915      0.944
  KNN                       0.888      0.923
  Naive Bayes               0.739      0.842
  Decision Tree             0.810      0.868
  AdaBoost                  0.908      0.938
  MLP                       0.902      0.932

  Meilleur modèle (F1) : Random Forest — F1 = 0.944


## 9. Évaluation par validation croisée et par sujet

Deux évaluations sont demandées :

### 10-fold cross-validation

Les segments sont répartis en 10 folds stratifiés. Cette évaluation est utile pour comparer les modèles, mais elle peut mélanger les sujets entre apprentissage et test.

### Leave-One-Subject-Out, LOSO

Un sujet est laissé de côté pour le test, tandis que le modèle est entraîné sur les autres sujets. Cette stratégie d'évaluation est plus réaliste, car elle permet de tester la capacité de généralisation du modèle sur un conducteur jamais vu auparavant. L'opération est ensuite répétée sur l'ensemble des sujets disponibles afin d'obtenir une évaluation plus robuste.

## Question

Pourquoi le LOSO est-il souvent plus difficile que le 10-fold classique ?

### Réponse

En **10-fold cross-validation**, toutes les fenêtres sont mélangées et réparties aléatoirement. Des fenêtres du *même sujet* se retrouvent à la fois dans le train et dans le test. Le modèle a donc « vu » les patterns propres à chaque individu pendant l'entraînement, ce qui facilite la prédiction.

En **LOSO**, le sujet de test est *entièrement absent* de l'entraînement. Le modèle doit généraliser à un nouveau conducteur jamais rencontré. C'est plus difficile pour deux raisons :

1. **Variabilité inter-sujets** : Les signatures EEG de la charge cognitive diffèrent entre personnes. Un pattern qui caractérise la charge élevée chez un sujet peut ne pas être visible chez un autre.

2. **Moins de données d'entraînement** : Avec N−1 sujets seulement, le modèle a moins d'exemples pour apprendre.

Le LOSO mesure donc la **généralisation à de nouveaux conducteurs**, ce qui est le vrai critère d'utilité pour une application réelle de détection de charge cognitive en conduite.

In [26]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score

logo = LeaveOneGroupOut()
unique_participants = np.unique(groups)

if len(unique_participants) < 2:
    print("LOSO nécessite au moins 2 participants.")
    print(f"Sujets disponibles : {unique_participants.tolist()}")
    print("Conseil : exécutez le batch TD3+TD4 pour tous les sujets, puis relancez ce TP.")
    results_loso = {}
else:
    results_loso = {}
    print("=== Leave-One-Subject-Out (LOSO) ===\n")
    print(f"  {'Modèle':<20} {'Accuracy':>10} {'F1-score':>10}")
    print("  " + "-" * 42)
    for name, pipe in pipelines.items():
        fold_accs, fold_f1s = [], []
        for train_idx, test_idx in logo.split(X, y, groups):
            X_tr, X_te = X[train_idx], X[test_idx]
            y_tr, y_te = y[train_idx], y[test_idx]
            if len(np.unique(y_tr)) < 2:
                continue
            pipe.fit(X_tr, y_tr)
            y_pred = pipe.predict(X_te)
            fold_accs.append(accuracy_score(y_te, y_pred))
            fold_f1s.append(f1_score(y_te, y_pred, zero_division=0))
        if fold_accs:
            results_loso[name] = {
                "accuracy": np.mean(fold_accs),
                "f1":       np.mean(fold_f1s),
            }
            print(f"  {name:<20} {results_loso[name]['accuracy']:>10.3f} "
                  f"{results_loso[name]['f1']:>10.3f}")

    if results_loso:
        best_loso = max(results_loso, key=lambda k: results_loso[k]["f1"])
        print(f"\n  Meilleur modèle LOSO (F1) : {best_loso} — F1 = {results_loso[best_loso]['f1']:.3f}")

LOSO nécessite au moins 2 participants.
Sujets disponibles : [1030]
Conseil : exécutez le batch TD3+TD4 pour tous les sujets, puis relancez ce TP.


## 10. Interprétation et discussion

Répondez aux questions suivantes dans le notebook :

1. Quel modèle obtient le meilleur F1-score en 10-fold ?
2. Quel modèle obtient le meilleur F1-score en LOSO ?
3. Les performances chutent-elles en LOSO ? Pourquoi ?
4. Les classes sont-elles équilibrées ?
5. Les résultats obtenus avec EEG seul vous semblent-ils suffisants pour une application réelle ?
6. Quelles limites voyez-vous à l'utilisation des labels subjectifs PAAS ?
7. Quelles améliorations proposeriez-vous ?

In [27]:
# Résumé des résultats pour faciliter l'interprétation

print("=== Comparaison 10-fold CV vs LOSO ===\n")
print(f"  {'Modèle':<20} {'CV F1':>8} {'LOSO F1':>9}")
print("  " + "-" * 40)
for name in classifiers:
    cv_f1   = results_cv.get(name, {}).get("f1", float("nan"))
    loso_f1 = results_loso.get(name, {}).get("f1", float("nan"))
    print(f"  {name:<20} {cv_f1:>8.3f} {loso_f1:>9.3f}")

if results_cv:
    best_cv = max(results_cv, key=lambda k: results_cv[k]["f1"])
    print(f"\n  Meilleur 10-fold : {best_cv} (F1={results_cv[best_cv]['f1']:.3f})")
if results_loso:
    best_loso = max(results_loso, key=lambda k: results_loso[k]["f1"])
    print(f"  Meilleur LOSO    : {best_loso} (F1={results_loso[best_loso]['f1']:.3f})")

print(f"\n  Distribution classes : {dict(zip(*np.unique(y, return_counts=True)))}")
total = len(y)
for cls, cnt in zip(*np.unique(y, return_counts=True)):
    label = "Faible" if cls == 0 else "Elevee"
    print(f"    Classe {cls} ({label}) : {cnt}/{total} = {100*cnt/total:.1f}%")

=== Comparaison 10-fold CV vs LOSO ===

  Modèle                  CV F1   LOSO F1
  ----------------------------------------
  LDA                     0.750       nan
  SVM                     0.921       nan
  Random Forest           0.944       nan
  KNN                     0.923       nan
  Naive Bayes             0.842       nan
  Decision Tree           0.868       nan
  AdaBoost                0.938       nan
  MLP                     0.932       nan

  Meilleur 10-fold : Random Forest (F1=0.944)

  Distribution classes : {np.int64(0): np.int64(37), np.int64(1): np.int64(105)}
    Classe 0 (Faible) : 37/142 = 26.1%
    Classe 1 (Elevee) : 105/142 = 73.9%


### Réponses — Interprétation et discussion

**1. Quel modèle obtient le meilleur F1-score en 10-fold ?**
→ À compléter après exécution (voir tableau ci-dessus). Souvent Random Forest ou SVM sur des features EEG.

**2. Quel modèle obtient le meilleur F1-score en LOSO ?**
→ À compléter après exécution. En général, les modèles plus robustes (LDA, SVM) tiennent mieux en LOSO que les modèles qui s'adaptent finement aux individus (Decision Tree, KNN).

**3. Les performances chutent-elles en LOSO ? Pourquoi ?**
→ Oui, en général les performances chutent en LOSO. La cause principale est la variabilité inter-sujets : les patterns EEG de la charge cognitive ne sont pas identiques d'une personne à l'autre. Le modèle apprend sur N−1 sujets et doit prédire pour un nouveau conducteur dont il n'a jamais vu les données.

**4. Les classes sont-elles équilibrées ?**
→ À vérifier dans la distribution ci-dessus. Si une classe domine (ex. charge faible beaucoup plus fréquente que charge élevée), le F1-score est plus informatif que l'accuracy, et des techniques comme le sur-échantillonnage (SMOTE) ou la pondération des classes (`class_weight='balanced'`) peuvent améliorer les résultats.

**5. Les résultats EEG seul vous semblent-ils suffisants pour une application réelle ?**
→ Probablement insuffisants avec un seul sujet. En situation réelle, on viserait F1 > 0.70 sur l'ensemble du dataset avec LOSO. L'EEG seul peut être suffisant si bien normalisé et combiné avec de l'adaptation au sujet.

**6. Quelles limites voyez-vous à l'utilisation des labels subjectifs PAAS ?**
→ Plusieurs limites : (a) Le PAAS est auto-rapporté toutes les 10 s — il y a une latence entre l'état réel et la déclaration. (b) La subjectivité varie : certains sujets notent plus sévèrement que d'autres. (c) Un seul chiffre résume une expérience multidimensionnelle (effort mental, arousal, stress).

**7. Quelles améliorations proposeriez-vous ?**
→ Pistes principales : (a) Normalisation par baseline individuelle activée (exécuter le batch complet TD3+TD4). (b) Sélection ou réduction de features (PCA, SelectKBest). (c) Adaptation au sujet (fine-tuning sur quelques fenêtres du nouveau conducteur). (d) Fusion multimodale (ECG + EDA). (e) Classifieurs séquentiels (LSTM, HMM) pour exploiter la dynamique temporelle.

## 11. Mini-système d'adaptation

À partir de la prédiction du modèle, on peut simuler une décision d'adaptation.

Exemple :

| Prédiction | Décision |
|---|---|
| charge faible | interface normale |
| charge élevée | simplification de l'interface |
| charge élevée persistante | alerte conducteur |

## Question

Pourquoi faut-il être prudent avant de déclencher une alerte sur une seule prédiction ?

### Réponse

Il faut être prudent pour trois raisons :

1. **Bruit et artefacts** : Une seule fenêtre de 10 secondes peut être contaminée par un artefact EEG momentané (clignement des yeux, contraction musculaire, choc de la route). La prédiction serait alors un faux positif sans rapport avec la charge réelle.

2. **Variabilité naturelle** : La charge cognitive fluctue normalement, et une brève élévation n'indique pas nécessairement un état persistant qui nécessite une intervention.

3. **Dégradation de l'expérience** : Déclencher des alertes trop fréquentes est contre-productif. Le conducteur sera distrait ou finira par ignorer les alertes (effet d'habituation). Il est plus fiable d'exiger un **lissage temporel** : ne déclencher une alerte que si plusieurs prédictions consécutives indiquent une charge élevée (ex. : 3 fenêtres consécutives = 30 secondes).

In [28]:
def decision_system(predictions, window_size=3):
    """
    Transforme des prédictions binaires en décisions d'adaptation.

    predictions : liste ou array (0 = charge faible, 1 = charge élevée)
    window_size : nb de fenêtres consécutives élevées pour déclencher une alerte

    Retourne une liste de chaînes de décision.
    """
    predictions = list(predictions)
    decisions = []
    for i, pred in enumerate(predictions):
        if pred == 0:
            decisions.append("interface normale")
        else:
            start = max(0, i - window_size + 1)
            recent = predictions[start : i + 1]
            if len(recent) == window_size and all(p == 1 for p in recent):
                decisions.append("alerte conducteur")
            else:
                decisions.append("simplification de l'interface")
    return decisions


# Démonstration sur un exemple fictif
exemple = [0, 0, 1, 0, 1, 1, 1, 1, 0, 1]
decisions = decision_system(exemple, window_size=3)

print(f"  {'Fen.':>5}  {'Prediction':<10}  Décision")
print("  " + "-" * 50)
for i, (pred, dec) in enumerate(zip(exemple, decisions)):
    label = "élevée " if pred == 1 else "faible "
    print(f"  {i:>5}  {label:<10}  {dec}")

   Fen.  Prediction  Décision
  --------------------------------------------------
      0  faible      interface normale
      1  faible      interface normale
      2  élevée      simplification de l'interface
      3  faible      interface normale
      4  élevée      simplification de l'interface
      5  élevée      simplification de l'interface
      6  élevée      alerte conducteur
      7  élevée      alerte conducteur
      8  faible      interface normale
      9  élevée      simplification de l'interface


## 12. Extension optionnelle — vers la multimodalité

Le cœur du TP est volontairement limité à l'EEG.

Une extension possible consiste à reproduire les mêmes étapes pour les autres modalités :

```text
ECG_Features  → Normalized_Features/ECG → Normalized_Features_With_Label/ECG
EDA_Features  → Normalized_Features/EDA → Normalized_Features_With_Label/EDA
Gaze_Features → Normalized_Features/Gaze → Normalized_Features_With_Label/Gaze
```

Puis à fusionner les features :

```text
EEG + ECG
EEG + EDA
EEG + Gaze
EEG + ECG + EDA + Gaze
```

La fusion la plus simple est une concaténation des colonnes de features pour des fenêtres correspondant au même sujet, au même niveau et au même indice de fenêtre.

## Question

Pourquoi la multimodalité peut-elle améliorer la détection de la charge cognitive ?

### Réponse

Chaque modalité capte un aspect différent de la charge cognitive :

- **EEG** : activité cérébrale directe, très sensible mais variable entre sujets et susceptible aux artefacts.
- **ECG** : fréquence cardiaque et variabilité (HRV). La charge élevée augmente la fréquence cardiaque et réduit la HRV.
- **EDA** : activité électrodermale (conductance cutanée), reflet de l'arousal et du stress. La charge cognitive déclenche une réponse sudorale mesurable.
- **Gaze** : attention visuelle (fixations, saccades, pupillométrie). La charge élevée modifie les patterns d'exploration visuelle.

En combinant ces sources, on bénéficie de deux avantages :

1. **Redondance** : si l'EEG est bruité pour un sujet donné (mauvais contact d'électrode), les autres modalités compensent.

2. **Complémentarité** : chaque modalité décrit un aspect différent — cognitif, autonome, comportemental. Ensemble, elles offrent une description plus complète de l'état du conducteur.

La littérature montre régulièrement que les systèmes multimodaux surpassent les systèmes unimodaux sur les tâches de reconnaissance d'état affectif et cognitif.